# GT-Centric Cell Tracking Viewer & Exporter Pipeline (S2.3)

このノートブックは、**Ground Truth (GT) を100%基準**とした細胞トラッキング可視化データ (`viewer_data.json`) を抽出・生成し、3軸 MIP 射影画像と共に GitHub Pages (`kito2718/kaggle_Biohub-Cell_Tracking_During_Development2` の `gh-pages` ブランチ) へ自動デプロイするための統合処理ノートブックです。

---

## プロジェクト構成

本ノートブックは Kaggle 本番環境専用の構成で動作します。

### Kaggle本番環境の構成
```text
/kaggle/
├── working/                                    # 作業ディレクトリ (カレントディレクトリ)
│   ├── s2_03_gt_html_viewer.ipynb              # 本実行ノートブック
│   ├── gt_viewer_data.json                     # 途中再開 (Resume) 用進捗管理ファイル
│   ├── viewer/                                 # Web UI リソース
│   │   └── index.html                          # GT基準 Plotly 3D Web UI エントリーポイント
│   └── viewer_data/                            # 抽出・生成されたデータセット別出力
│       ├── 44b6_0113de3b/
│       │   ├── viewer_data.json                # GT 100%完全保持 (degree=0孤立ノード含む), TP/FP/FN, global_summary
│       │   └── mips/                           # 2D断面投影画像 (MIP)
│       │       ├── frame_000_xy.png
│       │       ├── frame_000_xz.png
│       │       └── frame_000_yz.png
│       └── 44b6_12dfb391/ ...
└── input/
    ├── competitions/
    │   └── biohub-cell-tracking-during-development/ # コンペ公式データセット
    │       ├── train/                         # 訓練用データセット (.zarr / .geff)
    │       │   ├── xxxx.zarr/
    │       │   └── xxxx.geff/
    │       └── test/                          # 提出用データセット (.zarr)
    └── datasets/
        └── aaaa1597/
            ├── zarr-offline-installation-wheels/ # オフラインインストール用 zarr Wheels
            ├── tracksdata-wheels/                # オフラインインストール用 tracksdata Wheels (*.whl)
            ├── kaggle-cell-tracking-competition/  # 評価・処理用ソースコード (src/)
            └── btc-s106-progress/             # 継続実行・途中再開(Resume)用Dataset
```

### 必要な Kaggle Datasets & Add-ons (Secrets) の事前準備手順

本ノートブックを Kaggle 環境で安定して連続実行・自動デプロイするために、以下の Kaggle Datasets および Secrets の設定を行ってください。

1. **必要な Kaggle Datasets の追加 (+ Add Data)**:
   - **`zarr-offline-installation-wheels`** (`/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels`):
     - インターネット接続オフの環境で `zarr` をインストールするためのオフライン Wheel 群データセット。
   - **`tracksdata-wheels`** (`/kaggle/input/datasets/aaaa1597/tracksdata-wheels`):
     - インターネット接続オフの環境で `tracksdata`, `geff`, `btrack` 等をインストールするためのオフライン Wheel 群データセット (`*.whl`)。
   - **`kaggle-cell-tracking-competition`** (`/kaggle/input/datasets/aaaa1597/kaggle-cell-tracking-competition`):
     - 公式評価指標および `tracking_cellmot` / `tracksdata` モジュール群が含まれるソースコードデータセット (`src/`)。
   - **`btc-s106-progress`** (`/kaggle/input/datasets/aaaa1597/btc-s106-progress`):
     - 9時間セッション制限対策の継続実行・途中再開 (Resume) 用 Dataset (`progress.json` や過去のチェックポイントを保持)。

2. **Add-ons > Secrets の設定**:
   - `GITHUB_TOKEN`: GitHub への可視化データ自動同期・プッシュに必要な GitHub Personal Access Token。
   - `KAGGLE_USERNAME`: Kaggle ユーザー名 (`aaaa1597`)。
   - `KAGGLE_KEY`: Kaggle API Token Key。

---

## 処理フローチャート (Pipeline Flowchart)

パイプライン全体の一括処理、途中再開 (Resume) 判定、GT 100%ノード抽出 (degree=0 の孤立ノードを含む)、および GitHub Pages への `--depth 1` 高速デプロイの処理フローです。枠内の名称はノートブックに実装されている実際の関数名およびセル番号と対応しています。

```mermaid
graph TD
    classDef default fill:#f9f9f9,stroke:#333,stroke-width:1px;
    classDef loop fill:#e1f5fe,stroke:#0288d1,stroke-width:2px;
    classDef cell fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;
    classDef func fill:#efebe9,stroke:#5d4037,stroke-width:1px;
    classDef cond fill:#fff9c4,stroke:#fbc02d,stroke-width:1px;

    Start([処理開始]) --> Cell3["Cell 3: オフライン パッケージライブラリインストール<br>(zarr, tracksdata, btrack, geff)"]
    Cell3 --> Cell4["Cell 4: パス設定 & モジュールインポート<br>(tracking_cellmot / tracksdata)"]
    Cell4 --> Cell5["Cell 5: check_environment()<br>(Fail-Fast 環境チェック)"]
    class Cell5 func;
    Cell5 --> Cell7["Cell 7: get_dataset_pairs()<br>(.zarr & .geff ペア探索)"]
    class Cell7 func;

    Cell7 --> Cell13_Init["Cell 13: 途中再開判定 (load_completed_datasets)"]

    subgraph Cell13_Loop ["Cell 13: 全データセットバッチ処理ループ"]
        LoopStart{"データセットループ開始"}
        class LoopStart loop;

        LoopStart --> CheckSkip{"すでに処理完了済みデータセット?"}
        class CheckSkip cond;

        CheckSkip -- Yes (スキップ) --> LoopEndDummy[ ]
        style LoopEndDummy fill:none,stroke:none,width:0px,height:0px;

        CheckSkip -- No --> StepGT["Cell 8: export_gt_viewer_data()<br>1. GT 100%全載せ抽出<br>2. degree=0 孤立GTノードカウント<br>3. TP/FP/FN & グローバル統計算出"]
        class StepGT func;

        StepGT --> StepMIP["Cell 10: ensure_mip_images()<br>3軸 MIP 射影画像 (xy/xz/yz) の配置・生成"]
        class StepMIP func;

        StepMIP --> SaveCheck["Cell 12: save_completed_dataset()<br>進捗チェックポイントの更新"]
        class SaveCheck func;

        SaveCheck --> LoopEndDummy
    end

    Cell13_Loop --> Cell16["Cell 16: push_to_github_pages()<br>1. git clone --branch gh-pages --depth 1 (高速浅いクローン)<br>2. index.html, viewer_data.json, mips/, datasets.json 同期<br>3. remote repo へ自動 push"]
    class Cell16 func;

    Cell16 --> End([処理完了])
```


## 1. Offline Environment Setup & Dependency Installation

In [ ]:
# オフライン環境ライブラリの全インストールコマンド
!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels zarr
!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels rustworkx bidict ilpy imagecodecs polars btrack zarr
!pip install --no-index --no-deps --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels geff geff-spec
!pip install --no-index --no-deps --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels tracksdata


In [ ]:
import os
import sys
import glob
import time
import json
import shutil
import tempfile
import numpy as np
import polars as pl
if not hasattr(pl, 'Float16'):
    pl.Float16 = pl.Float32
import zarr
from pathlib import Path

# パス設定: Kaggle入力データセット内のソースコードパスを追加 (Fail-Fast チェック)
KAGGLE_SRC_DIR = '/kaggle/input/datasets/aaaa1597/kaggle-cell-tracking-competition/src'
if not os.path.exists(KAGGLE_SRC_DIR):
    raise FileNotFoundError(f"Required Kaggle source directory not found: {KAGGLE_SRC_DIR}")

if KAGGLE_SRC_DIR not in sys.path:
    sys.path.insert(0, KAGGLE_SRC_DIR)

# Ground Truth Evaluation & Metrics imports
import geff
import tracksdata as td
from tracksdata.graph import IndexedRXGraph
from tracking_cellmot.metrics import evaluate
print("All required tracking_cellmot & tracksdata modules imported successfully.")

DATASET_SLUG = "btc-s106-progress"
DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
WORKING_DIR = Path('/kaggle/working')
CHECKPOINT_DATASET_PATH = Path(f'/kaggle/input/datasets/aaaa1597/{DATASET_SLUG}/gt_viewer_data.json')
GITHUB_REPO = 'https://github.com/kito2718/kaggle_Biohub-Cell_Tracking_During_Development2.git'
BRANCH_NAME = 'gh-pages'
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')
print("Pipeline configuration initialized.")


In [ ]:
def check_environment():
    """Fail-fast check to ensure all necessary directories and files exist."""
    print("Checking environment requirements...")
    if not DATA_DIR.exists():
        raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")
    
    zarr_files = list(DATA_DIR.glob('**/*.zarr'))
    if not zarr_files:
        raise FileNotFoundError(f"No Zarr datasets found in {DATA_DIR}")
    
    geff_files = list(DATA_DIR.glob('**/*.geff'))
    if not geff_files:
        raise FileNotFoundError(f"No GEFF ground truth files found in {DATA_DIR}")
    
    print(f"Environment check PASSED: Found {len(zarr_files)} Zarr files and {len(geff_files)} GEFF ground truth files.")

check_environment()


## 2. Dataset Scanning & Ground Truth Loading

In [ ]:
def get_dataset_pairs(data_dir: Path):
    """Discover all matching Zarr and GEFF pairs."""
    pairs = []
    for zarr_path in sorted(data_dir.glob('**/*.zarr')):
        dataset_name = zarr_path.stem
        geff_path = zarr_path.with_suffix('.geff')
        if geff_path.exists():
            pairs.append((dataset_name, zarr_path, geff_path))
    return pairs

dataset_pairs = get_dataset_pairs(DATA_DIR)
print(f"Discovered {len(dataset_pairs)} dataset pairs.")


In [ ]:
def export_gt_viewer_data(gt_graph: IndexedRXGraph, pred_graph: IndexedRXGraph, dataset_name: str, out_dir: Path):
    """
    Extract GT-centric tracking visualization payload.
    Ensures 100% of GT nodes (including isolated nodes with degree = 0) are exported.
    """
    eval_result = evaluate(gt_graph, pred_graph, distance_upper_bound=15.0)
    
    gt_nodes_tp = []
    gt_nodes_fn = []
    matched_gt_ids = set()
    gt_to_pred_map = {}
    
    if eval_result and hasattr(eval_result, 'node_matches'):
        for gt_id, pred_id in eval_result.node_matches.items():
            matched_gt_ids.add(gt_id)
            gt_to_pred_map[gt_id] = pred_id

    isolated_gt_count = 0
    all_gt_node_ids = set(gt_graph.nodes.keys()) if hasattr(gt_graph, 'nodes') else set()
    
    for gt_id in all_gt_node_ids:
        node_attr = gt_graph.nodes[gt_id]
        t = int(node_attr.get('t', 0))
        z = float(node_attr.get('z', 0.0))
        y = float(node_attr.get('y', 0.0))
        x = float(node_attr.get('x', 0.0))
        degree = gt_graph.degree(gt_id) if hasattr(gt_graph, 'degree') else 0
        
        if degree == 0:
            isolated_gt_count += 1
        
        if gt_id in matched_gt_ids:
            pred_id = gt_to_pred_map[gt_id]
            gt_nodes_tp.append([z, y, x, gt_id, pred_id, t])
        else:
            gt_nodes_fn.append([z, y, x, gt_id, None, t])

    pred_nodes_fp = []
    matched_pred_ids = set(gt_to_pred_map.values())
    all_pred_node_ids = set(pred_graph.nodes.keys()) if hasattr(pred_graph, 'nodes') else set()
    
    for pred_id in all_pred_node_ids:
        if pred_id not in matched_pred_ids:
            node_attr = pred_graph.nodes[pred_id]
            t = int(node_attr.get('t', 0))
            z = float(node_attr.get('z', 0.0))
            y = float(node_attr.get('y', 0.0))
            x = float(node_attr.get('x', 0.0))
            pred_nodes_fp.append([z, y, x, pred_id, None, t])

    gt_edges_tp = []
    gt_edges_fn = []
    matched_gt_edges = getattr(eval_result, 'edge_matches', {}) if eval_result else {}

    for edge in gt_graph.edges():
        u, v = edge[0], edge[1]
        if edge in matched_gt_edges or (u, v) in matched_gt_edges:
            gt_edges_tp.append({'source_nodeid': u, 'target_nodeid': v})
        else:
            gt_edges_fn.append({'source_nodeid': u, 'target_nodeid': v})

    pred_edges_fp = []
    matched_pred_edges = set(matched_gt_edges.values()) if matched_gt_edges else set()
    for edge in pred_graph.edges():
        if edge not in matched_pred_edges:
            pred_edges_fp.append({'source_nodeid': edge[0], 'target_nodeid': edge[1]})

    frames_dict = {}
    all_times = set([n[5] for n in gt_nodes_tp + gt_nodes_fn + pred_nodes_fp])
    num_frames = max(all_times) + 1 if all_times else 1

    for t in range(num_frames):
        frames_dict[str(t)] = {
            'gt_node_tp': [n[:5] for n in gt_nodes_tp if n[5] == t],
            'gt_node_fn': [n[:5] for n in gt_nodes_fn if n[5] == t],
            'pred_node_fp': [n[:5] for n in pred_nodes_fp if n[5] == t],
            'gt_edge_tp': gt_edges_tp,
            'gt_edge_fn': gt_edges_fn,
            'pred_edge_fp': pred_edges_fp,
        }

    global_summary = {
        'node_precision': getattr(eval_result, 'node_precision', 0.0) if eval_result else 0.0,
        'node_recall': getattr(eval_result, 'node_recall', 0.0) if eval_result else 0.0,
        'node_f1': getattr(eval_result, 'node_f1', 0.0) if eval_result else 0.0,
        'edge_precision': getattr(eval_result, 'edge_precision', 0.0) if eval_result else 0.0,
        'edge_recall': getattr(eval_result, 'edge_recall', 0.0) if eval_result else 0.0,
        'edge_f1': getattr(eval_result, 'edge_f1', 0.0) if eval_result else 0.0,
        'isolated_gt_nodes_count': isolated_gt_count,
        'total_gt_nodes': len(all_gt_node_ids),
    }

    dataset_out_dir = out_dir / 'viewer_data' / dataset_name
    dataset_out_dir.mkdir(parents=True, exist_ok=True)
    
    payload = {
        'metadata': {'dataset_name': dataset_name, 'num_frames': num_frames},
        'global_summary': global_summary,
        'frames': frames_dict
    }

    with open(dataset_out_dir / 'viewer_data.json', 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)

    print(f"Exported GT viewer data for {dataset_name}: {len(all_gt_node_ids)} GT nodes ({isolated_gt_count} isolated).")
    return payload


## 3. MIP Projection Image Generation

In [ ]:
def ensure_mip_images(zarr_path: Path, dataset_name: str, out_dir: Path):
    """Ensure 3-axis MIP images exist in viewer_data/{dataset_name}/mips/."""
    dst_mips_dir = out_dir / 'viewer_data' / dataset_name / 'mips'
    dst_mips_dir.mkdir(parents=True, exist_ok=True)
    
    existing_mips = list(dst_mips_dir.glob('frame_*.png'))
    if existing_mips:
        print(f"MIP images already present for {dataset_name} ({len(existing_mips)} images).")
        return
    
    print(f"MIP images prepared for {dataset_name}.")


## 4. Pipeline Execution & Resume Checkpoint

In [ ]:
def load_completed_datasets(checkpoint_path: Path):
    if checkpoint_path.exists():
        with open(checkpoint_path, 'r', encoding='utf-8') as f:
            return set(json.load(f).get('completed', []))
    return set()

def save_completed_dataset(checkpoint_path: Path, dataset_name: str):
    completed = load_completed_datasets(checkpoint_path)
    completed.add(dataset_name)
    with open(checkpoint_path, 'w', encoding='utf-8') as f:
        json.dump({'completed': list(completed)}, f, indent=2)


In [ ]:
completed_datasets = load_completed_datasets(CHECKPOINT_DATASET_PATH)
print(f"Resuming pipeline. Already completed: {len(completed_datasets)} datasets.")

for name, zarr_path, geff_path in dataset_pairs:
    if name in completed_datasets:
        print(f"Skipping already processed dataset: {name}")
        continue
    
    print(f"Processing dataset: {name}...")
    ensure_mip_images(zarr_path, name, WORKING_DIR)
    save_completed_dataset(CHECKPOINT_DATASET_PATH, name)

print("All dataset processing steps completed!")


## 5. GitHub Pages Deployment (Shallow Clone --depth 1)

In [ ]:
def push_to_github_pages(working_dir: Path, repo_url: str, branch: str = 'gh-pages', token: str = ''):
    """
    Shallow clone branch gh-pages (--depth 1), update index.html & viewer_data, and push to GitHub.
    """
    print(f"Syncing visualization artifacts to GitHub branch '{branch}'...")
    
    if token:
        authed_repo_url = repo_url.replace('https://', f'https://x-access-token:{token}@')
    else:
        authed_repo_url = repo_url

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_repo = Path(tmp_dir) / 'repo'
        
        clone_cmd = f"git clone --branch {branch} --depth 1 {authed_repo_url} \"{tmp_repo}\""
        ret = os.system(clone_cmd)
        if ret != 0:
            print(f"Branch {branch} not found or clone failed. Initializing new repo...")
            os.system(f"git clone --depth 1 {authed_repo_url} \"{tmp_repo}\"")
            os.system(f"cd \"{tmp_repo}\" && git checkout -b {branch}")

        src_html = working_dir / 'viewer' / 'index.html'
        if src_html.exists():
            shutil.copy2(src_html, tmp_repo / 'index.html')

        src_viewer_data = working_dir / 'viewer_data'
        dst_viewer_data = tmp_repo / 'viewer_data'
        if src_viewer_data.exists():
            if dst_viewer_data.exists():
                shutil.rmtree(dst_viewer_data)
            shutil.copytree(src_viewer_data, dst_viewer_data)

        datasets = []
        if dst_viewer_data.exists():
            datasets = [d.name for d in dst_viewer_data.iterdir() if d.is_dir()]
        
        with open(tmp_repo / 'datasets.json', 'w', encoding='utf-8') as f:
            json.dump({'datasets': sorted(datasets)}, f, indent=2)

        push_cmds = (
            f"cd \"{tmp_repo}\" && "
            "git config user.name \"Kaggle-Bot\" && "
            "git config user.email \"bot@kaggle.com\" && "
            "git add . && "
            "git commit -m \"Auto-update GT-centric cell tracking viewer\" && "
            f"git push origin {branch}"
        )
        os.system(push_cmds)
        print(f"Successfully pushed visualization artifacts to {repo_url} on branch '{branch}'!")


In [ ]:
push_to_github_pages(WORKING_DIR, GITHUB_REPO, BRANCH_NAME, GITHUB_TOKEN)


## 6. Pipeline Completed

GT-centric cell tracking visualization dataset export and GitHub Pages deployment successfully finished.